# Stage 2: Lin-KK Quality

Validates each spectrum by Kramers-Kronig compliance and picks the best replica per (condition, T).

**Reads:** `{sample_id}/Results/{condition}/stage1_labeling.xlsx` · `{sample_id}/ISM validation/*.ism`
**Writes:** `{sample_id}/Results/{condition}/stage2_kk.xlsx`

Set `FOCUS_T` to process one temperature at a time. The export is merge-aware: other temperatures are preserved in `stage2_kk.xlsx`.

## Quick links
- [Configuration](#configuration): sample_id, KK parameters, KK_OVERRIDES, OVERRIDES
- [Condition selector](#condition-selector): which conditions and focus temperature to process
- [Step 1: Batch KK](#step-1-batch-kk): silent run and GREEN/YELLOW/RED classification table
- [Step 2: Tune flagged spectra](#step-2-tune-flagged-spectra): interactive panel for YELLOW/RED
- [Step 3: Export](#step-3-export): selection summary and stage2_kk.xlsx export

## Configuration

In [ ]:
import json
from pathlib import Path

_cfg      = json.loads(Path("session.json").read_text()) if Path("session.json").exists() else {}
sample_id = _cfg.get("sample_id") or input("Sample folder name: ").strip()

# conditions saved in stage 0; leave empty to process all
_saved           = _cfg.get("conditions", [])
condition_filter = _saved

FOCUS_CONDITION = None
FOCUS_T         = None
SKIP_EXISTING   = False

# Lin-KK parameters (calibrated on this dataset; change only to test literature defaults)
KK_USE_BINARY_M  = False   # linear search: reproducible, RelaxIS default
KK_MU_TARGET     = 0.50    # sign-change fraction target (RelaxIS default)
KK_C             = 0.76    # M/N ratio (calibrated); Schonleber 2014 c=0.85 is a mu threshold, not M/N
KK_IQR_FENCE     = 0.5     # IQR fence multiplier; lower = stricter edge cut
KK_IQR_WINDOW    = 10      # consecutive clean points to confirm the cut edge
KK_F_MIN_HARD    = 50      # [Hz] hard lower limit (LF electrode noise)
KK_F_MAX_HARD    = None    # [Hz] None = adaptive IQR picks the upper limit
KK_USE_W_CRITERIA = False  # True = ceramic-aware: W_re>=0.95 AND W_im>=0.93

# Cell B generates KK_OVERRIDES automatically after Step 1.
# The replica selector in Step 2 writes OVERRIDES without manual editing.
KK_OVERRIDES = {}
OVERRIDES    = {}

## Import

In [ ]:
import sys
import gc
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism, load_csv_spectrum, scan_input_spectra
from pipeline.quality import (
    run_linkk, select_best_replica, compute_frequency_cutoffs, kk_summary_table,
    strip_inductive,
)
from pipeline.plots import apply_pub_style, COLOR_MAP

apply_pub_style()

sample_dir   = NOTEBOOK_DIR / sample_id
results_base = sample_dir / "Results"

# detect entry mode: input_spectra/ (CSV/TXT) or normal stage1_labeling.xlsx
_df_csv = scan_input_spectra(sample_dir)
_CSV_MODE = _df_csv is not None

if _CSV_MODE:
    print(f"Entry mode: input_spectra (CSV/TXT)")
    all_conditions = sorted(_df_csv["condition"].unique().tolist())
else:
    all_conditions = sorted([
        d.name for d in results_base.iterdir()
        if d.is_dir() and (d / "stage1_labeling.xlsx").exists()
    ])

conditions = [c for c in all_conditions if c in condition_filter] if condition_filter else all_conditions[:]
if FOCUS_CONDITION is not None:
    conditions = [c for c in conditions if c == FOCUS_CONDITION]
    print(f"FOCUS_CONDITION active: {FOCUS_CONDITION}")
if SKIP_EXISTING:
    skipped    = [c for c in conditions if (results_base / c / "stage2_kk.xlsx").exists()]
    conditions = [c for c in conditions if not (results_base / c / "stage2_kk.xlsx").exists()]
    if skipped:
        print(f"SKIP_EXISTING=True: skipping {len(skipped)} already computed condition(s)")

_LABELED_RE = re.compile(r'_\d{2,4}[Cc](?:_\d+)?\.[ci]', re.IGNORECASE)


def _short_cond(name: str) -> str:
    stripped = name[len(sample_id):].lstrip("_") if name.startswith(sample_id) else name
    parts = stripped.split("_")
    if parts and len(parts[0]) <= 3 and not re.match(r"^(Ar|O2|N2|H2)", parts[0], re.I):
        stripped = "_".join(parts[1:])
    parts = stripped.split("_")
    if len(parts) >= 4 and parts[-3].isdigit() and parts[-2].isdigit():
        t_hi, t_lo = parts[-3], parts[-2]
        gas = " ".join(parts[:-3])
        return f"{gas} | {t_lo}-{t_hi}C"
    return stripped


n_ism_total = 0
if not _CSV_MODE:
    for c in conditions:
        xlsx_path = results_base / c / "stage1_labeling.xlsx"
        if xlsx_path.exists():
            try:
                df_tmp = pd.read_excel(xlsx_path, sheet_name="VALID")
                n_ism_total += df_tmp["file"].apply(lambda f: bool(_LABELED_RE.search(str(f)))).sum()
            except Exception:
                pass
else:
    n_ism_total = len(_df_csv[_df_csv["condition"].isin(conditions)])

print(f"Sample     : {sample_id}")
print(f"Conditions : {len(conditions)}")
for c in conditions:
    print(f"  {_short_cond(c)}")
print(f"Estimated  : {n_ism_total} spectra x ~20 ms = {n_ism_total * 0.020:.0f} s")

_STATUS_ICONS = {"GREEN": "✓ GREEN", "YELLOW": "⚠ YELLOW", "RED": "✗ RED"}

## Condition selector

Toggle FOCUS to restrict processing to one condition and/or one temperature.
Leave FOCUS off to process all conditions from `condition_filter`.

In [ ]:
NOTEBOOK_DIR = Path().resolve()
from pipeline.interactive import discover_conditions, make_focus_panel


def _apply_focus_nb02(cond, T):
    global FOCUS_CONDITION, FOCUS_T
    FOCUS_CONDITION = cond
    FOCUS_T = T


make_focus_panel(
    conditions = discover_conditions(NOTEBOOK_DIR / sample_id, require="stage1_labeling.xlsx"),
    temps      = [600, 575, 550, 525, 500, 475, 450, 425, 400],
    set_focus  = _apply_focus_nb02,
    init_cond  = FOCUS_CONDITION,
    init_T     = FOCUS_T,
)

## Step 1: Batch KK

Runs Lin-KK on every (condition, T) and classifies each spectrum GREEN / YELLOW / RED.
If any YELLOW or RED appear: Cell B suggests `KK_OVERRIDES`, click Apply, then re-run this cell.
When all GREEN: skip Step 2 and go directly to Step 3.

In [ ]:
# Cell A; silent batch run + classification table

# read from condition selector widget if available
_gb = globals()
if '_w_conds' in _gb:
    conditions      = list(_gb['_w_conds'].value)
    FOCUS_T         = _gb['_w_focusT'].value
    FOCUS_CONDITION = _gb['_w_focusC'].value
    if FOCUS_CONDITION is not None:
        conditions = [c for c in conditions if c == FOCUS_CONDITION]


def _resolve_cutoffs(condition: str, T_int: int) -> tuple:
    cond_ov = KK_OVERRIDES.get(condition, {})
    t_ov    = cond_ov.get(T_int, {})
    f_min_h = t_ov.get("f_min_hard", cond_ov.get("f_min_hard", KK_F_MIN_HARD))
    f_max_h = t_ov.get("f_max_hard", cond_ov.get("f_max_hard", KK_F_MAX_HARD))
    return f_min_h, f_max_h


def _n_in_hard_window(freq_b: np.ndarray, f_min_h, f_max_h) -> int:
    lo = f_min_h if f_min_h is not None else float(freq_b.min())
    hi = f_max_h if f_max_h is not None else float(freq_b.max())
    n  = int(((freq_b >= lo) & (freq_b <= hi)).sum())
    return n if n > 0 else len(freq_b)


def _classify_kk(kk_score: float, n_kept: int, n_total: int,
                 W_re: float | None = None, W_im: float | None = None) -> str:
    frac_cut = 1.0 - (n_kept / n_total) if n_total > 0 else 1.0
    if KK_USE_W_CRITERIA and (W_re is not None) and (W_im is not None):
        if W_re >= 0.95 and W_im >= 0.93 and frac_cut <= 0.20:
            return "GREEN"
        if W_re >= 0.90 and W_im >= 0.88 and frac_cut <= 0.40:
            return "YELLOW"
        return "RED"
    if kk_score >= 0.97 and frac_cut <= 0.20:
        return "GREEN"
    if kk_score >= 0.90 and frac_cut <= 0.40:
        return "YELLOW"
    return "RED"


if FOCUS_T is not None:
    print(f"FOCUS_T = {FOCUS_T} C: processing only T={FOCUS_T}C across all conditions")

all_kk_data = {}
_kk_class   = {}

for condition in tqdm(conditions, desc="Conditions", unit="cond"):

    if _CSV_MODE:
        df_cond   = _df_csv[_df_csv["condition"] == condition].copy()
        t_groups  = sorted(df_cond["T_nominal"].dropna().unique())
        input_dir = sample_dir / "input_spectra" / condition
    else:
        xlsx_path     = sample_dir / "Results" / condition / "stage1_labeling.xlsx"
        df_stage1_all = pd.read_excel(xlsx_path, sheet_name="VALID")
        _RE           = re.compile(r'_\d{2,4}[Cc](?:_\d+)?\.ism$', re.IGNORECASE)
        labeled_mask  = df_stage1_all["file"].apply(lambda f: bool(_RE.search(str(f))))
        df_cond       = df_stage1_all[labeled_mask].copy()
        t_groups      = sorted(df_cond["T_nominal"].dropna().unique())
        input_dir     = sample_dir / "ISM validation" / condition

    cond_data  = {}
    cond_class = {}

    for T in t_groups:
        T_int = int(T)
        if FOCUS_T is not None and T_int != FOCUS_T:
            continue

        f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
        group_df         = df_cond[df_cond["T_nominal"] == T].sort_values("replica")

        records = []
        for _, row in group_df.iterrows():
            fpath = Path(row["full_path"]) if "full_path" in row and pd.notna(row.get("full_path")) else input_dir / row["file"]
            if not fpath.exists():
                continue
            if _CSV_MODE:
                rec = load_csv_spectrum(fpath)
            else:
                rec = load_ism(fpath)
            rec.T_nominal = T
            rec.T_mean    = row.get("T_mean")
            rec.pO2_mean  = row.get("pO2_mean")
            rec.replica   = row.get("replica")
            records.append(rec)

        if not records:
            continue

        kk_results = []
        for rec in records:
            freq_c, Z_re_c, Z_im_c, _ = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
            res = run_linkk(
                freq_c, Z_re_c, Z_im_c,
                c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
                iqr_fence_factor=KK_IQR_FENCE, iqr_window=KK_IQR_WINDOW,
                f_min_hard=f_min_h, f_max_hard=f_max_h,
            )
            kk_results.append(res)

        best_idx = select_best_replica(kk_results)
        override_val = (OVERRIDES.get(condition, {}).get(T_int) or
                        OVERRIDES.get(condition, {}).get(T))
        if override_val:
            names = [r.path.name for r in records]
            if isinstance(override_val, list):
                compare_idx = [names.index(f) for f in override_val if f in names]
                if compare_idx:
                    best_idx = max(compare_idx, key=lambda i: kk_results[i]["kk_score"])
            elif override_val in names:
                best_idx = names.index(override_val)

        best     = kk_results[best_idx]
        freq_b   = best["freq"]
        n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
        n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
        cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                W_re=best["W_re"], W_im=best["W_im"])

        cond_data[T_int] = {
            "records":       records,
            "kk_results":    kk_results,
            "best_idx":      best_idx,
            "selected_file": records[best_idx].path.name,
            "f_min_cut":     best["f_min_cut"],
            "f_max_cut":     best["f_max_cut"],
            "df_summary":    kk_summary_table(records, kk_results, best_idx),
        }
        cond_class[T_int] = cls

    all_kk_data[condition] = cond_data
    _kk_class[condition]   = cond_class
    gc.collect()


_STATUS_COLORS = {"GREEN": "#d4edda", "YELLOW": "#fff3cd", "RED": "#f8d7da"}


def _build_summary_table():
    rows = []
    for condition in conditions:
        for T_int in sorted(_kk_class.get(condition, {}).keys(), reverse=True):
            data     = all_kk_data[condition][T_int]
            best     = data["kk_results"][data["best_idx"]]
            freq_b   = best["freq"]
            f_min_h, f_max_h = _resolve_cutoffs(condition, T_int)
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            frac_cut_pct = round((1.0 - n_kept / n_window) * 100, 1) if n_window else 100.0
            rows.append({
                "condition":  _short_cond(condition),
                "T [°C]":     T_int,
                "file":       data["selected_file"],
                "kk_score":   round(best["kk_score"], 3),
                "W_re":       round(best["W_re"], 3),
                "W_im":       round(best["W_im"], 3),
                "f_min [Hz]": (round(best["f_min_cut"], 1)
                               if best["f_min_cut"] is not None else None),
                "f_max [Hz]": (round(best["f_max_cut"], 1)
                               if best["f_max_cut"] is not None else None),
                "% cut":      frac_cut_pct,
                "STATUS":     _STATUS_ICONS[_kk_class[condition][T_int]],
            })
    return pd.DataFrame(rows)


def _hl_status(val):
    for k, lbl in _STATUS_ICONS.items():
        if val == lbl:
            return f"background-color: {_STATUS_COLORS[k]}"
    return ""


df_summary = _build_summary_table()
styler = (
    df_summary.style
    .map(_hl_status, subset=["STATUS"])
    .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}", "% cut": "{:.1f}"})
    .hide(axis="index")
)
display(styler)

n_green  = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
n_yellow = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
n_red    = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
n_total  = n_green + n_yellow + n_red
t_label  = f" (FOCUS_T={FOCUS_T}C)" if FOCUS_T is not None else ""
crit_lbl = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score>=0.97 (strict)"
print(f"\n{n_total} spectra{t_label}: {n_green} GREEN | {n_yellow} YELLOW | {n_red} RED  [{crit_lbl}]")
if n_yellow + n_red > 0:
    print("Check the suggested KK_OVERRIDES in the next cell, then run Step 2.")
else:
    print("All clean. Skip Step 2 and go straight to Step 3 (export).")

In [ ]:
# Cell B; suggested KK_OVERRIDES (auto-generated from flagged spectra above)

_inv = {v: k for k, v in _STATUS_ICONS.items()}

needs = {
    cond: {
        T_int: {
            "f_min_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_min_cut"]), 2),
            "f_max_hard": round(float(
                all_kk_data[cond][T_int]["kk_results"][
                    all_kk_data[cond][T_int]["best_idx"]
                ]["f_max_cut"]), 2),
        }
        for T_int, cls in T_dict.items()
        if cls in ("YELLOW", "RED")
    }
    for cond, T_dict in _kk_class.items()
    if any(cls in ("YELLOW", "RED") for cls in T_dict.values())
}

if not needs:
    print("All spectra GREEN; no overrides needed. Go to Step 3.")
else:
    print("# Suggested KK_OVERRIDES; paste into Step 2 below, adjust if needed")
    print("KK_OVERRIDES = {")
    for cond, T_dict in needs.items():
        print(f'    "{cond}": {{')
        for T_int, ov in sorted(T_dict.items(), reverse=True):
            tag = _kk_class[cond][T_int]
            print(f'        {T_int}: {{"f_min_hard": {ov["f_min_hard"]}, '
                  f'"f_max_hard": {ov["f_max_hard"]}}},  # {tag}')
        print("    },")
    print("}")

    # One-click apply (no copy-paste): merge suggestions into live KK_OVERRIDES.
    try:
        import ipywidgets as _W
        from IPython.display import display as _disp
        _btn = _W.Button(description="📥 Apply suggested overrides", button_style="warning",
                         layout=_W.Layout(width="300px"),
                         tooltip="Merge the suggestions above into the in-memory KK_OVERRIDES, then re-run Step 1")
        _msg = _W.HTML()
        def _apply_kk(_b):
            n = 0
            for _c, _td in needs.items():
                KK_OVERRIDES.setdefault(_c, {}).update(_td); n += len(_td)
            _msg.value = (f"<b style='color:#b36b00'>Applied {n} override(s)</b> to KK_OVERRIDES "
                          ", re-run Step 1 (batch KK) to use them.")
        _btn.on_click(_apply_kk)
        _disp(_W.VBox([_btn, _msg]))
    except Exception as _e:
        print(f"[INFO] apply button needs ipywidgets ({_e}).")

## Step 2: Tune flagged spectra

Use the panel below. Dropdowns open on the worst-status spectrum first.
Move `f_min` / `f_max` sliders until residuals flatten; use Replica to compare or force a specific file.

**Workflow:** Retest KK updates the counters at the bottom of the panel (✓ ⚠ ✗) in real time.
To see the changes in the Step 1 colour table, re-run Cell A after finishing the tuning.

In [ ]:
# Quick KK tuning panel; re-run Lin-KK for one (condition, T) with custom f_min / f_max.
# Updates KK_OVERRIDES and OVERRIDES in memory; re-run Step 3 to persist.
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS_NB02 = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); KK tuning panel disabled.")
    _HAS_WIDGETS_NB02 = False


def _reclassify_all() -> None:
    global _kk_class
    for cond, td in all_kk_data.items():
        for T_int, data in td.items():
            best     = data["kk_results"][data["best_idx"]]
            freq_b   = best["freq"]
            f_min_h, f_max_h = _resolve_cutoffs(cond, T_int)
            n_window = _n_in_hard_window(freq_b, f_min_h, f_max_h)
            n_kept   = int(((freq_b >= best["f_min_cut"]) & (freq_b <= best["f_max_cut"])).sum())
            cls      = _classify_kk(best["kk_score"], n_kept, n_window,
                                    W_re=best["W_re"], W_im=best["W_im"])
            _kk_class.setdefault(cond, {})[T_int] = cls


def _count_status() -> tuple[int, int, int]:
    g = sum(v == "GREEN"  for d in _kk_class.values() for v in d.values())
    y = sum(v == "YELLOW" for d in _kk_class.values() for v in d.values())
    r = sum(v == "RED"    for d in _kk_class.values() for v in d.values())
    return g, y, r


def _status_chips_html() -> str:
    g, y, r = _count_status()
    crit = "W-criteria (ceramic electrolyte)" if KK_USE_W_CRITERIA else "kk_score>=0.97 (strict)"
    return (f"<div style='font-size:13px; padding:4px 0'>"
            f"<span style='background:#d4edda; padding:3px 8px; border-radius:4px'>✓ {g}</span>&nbsp; "
            f"<span style='background:#fff3cd; padding:3px 8px; border-radius:4px'>⚠ {y}</span>&nbsp; "
            f"<span style='background:#f8d7da; padding:3px 8px; border-radius:4px'>✗ {r}</span>&nbsp;&nbsp; "
            f"<i>criterion: {crit}</i></div>")


_ST_ORDER = {"RED": 0, "YELLOW": 1, "GREEN": 2}
_ST_ICON  = {"RED": "✗", "YELLOW": "⚠", "GREEN": "✓"}


def _cond_worst(cond: str) -> str:
    vals = list(_kk_class.get(cond, {}).values())
    if "RED"    in vals: return "RED"
    if "YELLOW" in vals: return "YELLOW"
    return "GREEN"


def _cond_options() -> list:
    conds = [c for c, d in all_kk_data.items() if d]
    conds.sort(key=lambda c: _ST_ORDER[_cond_worst(c)])
    return [(f"{_ST_ICON[_cond_worst(c)]}  {_short_cond(c)}", c) for c in conds]


def _T_options(cond: str) -> list:
    ts = sorted(all_kk_data.get(cond, {}).keys(), reverse=True)
    ts.sort(key=lambda t: _ST_ORDER[_kk_class.get(cond, {}).get(t, "GREEN")])
    return [(f"{_ST_ICON[_kk_class.get(cond, {}).get(t, 'GREEN')]}  {t} °C", t) for t in ts]


def _replica_options(cond: str, T: int) -> list:
    data = all_kk_data.get(cond, {}).get(T)
    if not data:
        return [("(no data)", None)]
    best_idx = data["best_idx"]
    forced   = OVERRIDES.get(cond, {}).get(T)
    return [
        (f"{rec.path.name}"
         f"{'  (auto)' if i == best_idx and not forced else ''}"
         f"{'  (forced)' if forced and rec.path.name == forced else ''}",
         rec.path.name)
        for i, rec in enumerate(data["records"])
    ]


def _retest_kk(condition: str, T_int: int, f_min: float | None, f_max: float | None,
               fence: float | None = None, window: int | None = None) -> None:
    data = all_kk_data.get(condition, {}).get(T_int)
    if data is None:
        print(f"[WARN] no KK data for {condition} T={T_int}; run Step 1 first.")
        return
    rec = data["records"][data["best_idx"]]

    if f_min is not None or f_max is not None:
        KK_OVERRIDES.setdefault(condition, {}).setdefault(T_int, {})
        if f_min is not None: KK_OVERRIDES[condition][T_int]["f_min_hard"] = f_min
        if f_max is not None: KK_OVERRIDES[condition][T_int]["f_max_hard"] = f_max

    freq_c, Z_re_c, Z_im_c, n_ind = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
    res = run_linkk(
        freq_c, Z_re_c, Z_im_c,
        c=KK_C, use_binary_M=KK_USE_BINARY_M, mu_target=KK_MU_TARGET,
        iqr_fence_factor=(fence if fence is not None else KK_IQR_FENCE),
        iqr_window=(window if window is not None else KK_IQR_WINDOW),
        f_min_hard=f_min, f_max_hard=f_max,
    )
    freq_b = res["freq"]
    n_win  = _n_in_hard_window(freq_b, f_min, f_max)
    n_kept = int(((freq_b >= res["f_min_cut"]) & (freq_b <= res["f_max_cut"])).sum())
    cls    = _classify_kk(res["kk_score"], n_kept, n_win, W_re=res["W_re"], W_im=res["W_im"])
    print(f"  kk={res['kk_score']:.4f}  W_re={res['W_re']:.3f}  W_im={res['W_im']:.3f}  "
          f"M={res['M']}  mu={res['mu']:.2f}  [{cls}]  "
          f"f_min_cut={res['f_min_cut']:.1f} Hz  f_max_cut={res['f_max_cut']:.1f} Hz")

    fig, ax = plt.subplots(figsize=(8, 3.2))
    color    = COLOR_MAP.get(T_int, "#555555")
    fence_pc = res["cutoff_fence"] * 100
    ax.semilogx(freq_b, res["res_re"]*100, "o-", ms=2.5, lw=0.8, color=color, label="Re")
    ax.semilogx(freq_b, res["res_im"]*100, "o-", ms=2.5, lw=0.8, color="darkorange", label="Im")
    ax.axhline( fence_pc, color="steelblue", ls="--", lw=0.9, label=f"+/-{fence_pc:.1f}%")
    ax.axhline(-fence_pc, color="steelblue", ls="--", lw=0.9)
    ax.axhline(0, color="k", lw=0.4)
    if res["f_min_cut"]: ax.axvline(res["f_min_cut"], color="grey", ls="--", lw=1.0)
    if res["f_max_cut"]: ax.axvline(res["f_max_cut"], color="grey", ls="-.", lw=1.0)
    ax.set_xlabel("Frequency [Hz]"); ax.set_ylabel("Residual / |Z| [%]")
    ax.set_title(f"{condition}  |  T={T_int} C  |  panel re-test  [{cls}]", fontsize=9)
    ax.grid(True, which="both", ls=":", alpha=0.3)
    ax.legend(fontsize=7, frameon=False, loc="upper left")
    plt.tight_layout()
    plt.show()


_KK_PRESETS = {
    "Conservative (publication)": dict(
        KK_C=0.85, KK_IQR_FENCE=2.0, KK_IQR_WINDOW=5,
        KK_F_MIN_HARD=30, KK_USE_W_CRITERIA=False),
    "Standard (current optimum)": dict(
        KK_C=0.76, KK_IQR_FENCE=0.5, KK_IQR_WINDOW=10,
        KK_F_MIN_HARD=50, KK_USE_W_CRITERIA=False),
    "Permissive (ceramic electrolyte-aware)": dict(
        KK_C=0.76, KK_IQR_FENCE=0.5, KK_IQR_WINDOW=10,
        KK_F_MIN_HARD=50, KK_USE_W_CRITERIA=True),
    "RelaxIS defaults": dict(
        KK_C=0.85, KK_IQR_FENCE=2.0, KK_IQR_WINDOW=5,
        KK_F_MIN_HARD=None, KK_USE_W_CRITERIA=False),
}


if _HAS_WIDGETS_NB02 and all_kk_data:
    _cond_opts = _cond_options()
    if _cond_opts:
        _c0_2   = _cond_opts[0][1]
        _T_opts = _T_options(_c0_2)
        _T0_2   = _T_opts[0][1] if _T_opts else 600

        wc  = W.Dropdown(options=_cond_opts, value=_c0_2, description="Cond:",
                         layout=W.Layout(width="420px"))
        wT  = W.Dropdown(options=_T_opts, value=_T0_2, description="T [C]:",
                         layout=W.Layout(width="220px"))
        wmn = W.FloatLogSlider(value=KK_F_MIN_HARD or 30, base=10, min=0, max=6, step=0.02,
                               description="f_min Hz", readout_format=".1f",
                               continuous_update=False, layout=W.Layout(width="300px"))
        wmn_txt = W.BoundedFloatText(value=KK_F_MIN_HARD or 30, min=0.1, max=1e7, step=1,
                                     layout=W.Layout(width="110px"),
                                     tooltip="type exact f_min [Hz]")
        wmx = W.FloatLogSlider(value=KK_F_MAX_HARD or 1e6, base=10, min=2, max=8, step=0.02,
                               description="f_max Hz", readout_format=".1e",
                               continuous_update=False, layout=W.Layout(width="300px"))
        wmx_txt = W.BoundedFloatText(value=KK_F_MAX_HARD or 1e6, min=1, max=1e8, step=1,
                                     layout=W.Layout(width="110px"),
                                     tooltip="type exact f_max [Hz]")
        wfence = W.FloatSlider(value=KK_IQR_FENCE, min=0.1, max=3.0, step=0.1,
                               description="fence", readout_format=".1f",
                               continuous_update=False, layout=W.Layout(width="360px"),
                               tooltip="IQR fence multiplier: lower = stricter edge cut")
        wwin = W.IntSlider(value=KK_IQR_WINDOW, min=3, max=20, step=1,
                           description="window", continuous_update=False,
                           layout=W.Layout(width="300px"),
                           tooltip="consecutive clean points to confirm the cut edge")
        w_crit = W.ToggleButton(
            value=bool(KK_USE_W_CRITERIA),
            description=("ceramic W-criteria" if KK_USE_W_CRITERIA else "strict kk>=0.97"),
            button_style="info", layout=W.Layout(width="170px"),
            tooltip="Click to switch classification criterion")
        wgo = W.Button(description="Retest KK", button_style="primary",
                       layout=W.Layout(width="160px"))

        _w_replica = W.Dropdown(
            options=_replica_options(_c0_2, _T0_2),
            description="Replica:",
            layout=W.Layout(width="500px"),
            style={"description_width": "60px"}
        )
        _w_force = W.Button(
            description="Force this replica",
            layout=W.Layout(width="185px"),
            tooltip="Add to OVERRIDES; re-run Step 3 to export this file"
        )

        wpreset = W.Dropdown(options=list(_KK_PRESETS), value="Standard (current optimum)",
                             description="Preset:", layout=W.Layout(width="380px"))
        w_apply = W.Button(description="Apply preset", button_style="warning",
                           layout=W.Layout(width="160px"))

        chips = W.HTML(value=_status_chips_html())
        out2  = W.Output()

        _syncing = [False]

        def _wmn_slider_to_txt(change):
            if not _syncing[0]:
                _syncing[0] = True
                wmn_txt.value = round(change["new"], 2)
                _syncing[0] = False
        def _wmn_txt_to_slider(change):
            if not _syncing[0] and change["new"] > 0:
                _syncing[0] = True
                wmn.value = change["new"]
                _syncing[0] = False
        def _wmx_slider_to_txt(change):
            if not _syncing[0]:
                _syncing[0] = True
                wmx_txt.value = round(change["new"], 2)
                _syncing[0] = False
        def _wmx_txt_to_slider(change):
            if not _syncing[0] and change["new"] > 0:
                _syncing[0] = True
                wmx.value = change["new"]
                _syncing[0] = False
        wmn.observe(_wmn_slider_to_txt, names="value")
        wmn_txt.observe(_wmn_txt_to_slider, names="value")
        wmx.observe(_wmx_slider_to_txt, names="value")
        wmx_txt.observe(_wmx_txt_to_slider, names="value")

        def _load_state_into_widgets(*_):
            cond, T = wc.value, int(wT.value)
            ov   = KK_OVERRIDES.get(cond, {})
            t_ov = ov.get(T, {}) if isinstance(ov.get(T), dict) else {}
            fmin = t_ov.get("f_min_hard",
                            ov.get("f_min_hard") if not isinstance(ov.get("f_min_hard"), dict) else None)
            fmax = t_ov.get("f_max_hard",
                            ov.get("f_max_hard") if not isinstance(ov.get("f_max_hard"), dict) else None)
            wmn.value = float(fmin) if fmin is not None else (KK_F_MIN_HARD or 30)
            wmx.value = float(fmax) if fmax is not None else (KK_F_MAX_HARD or 1e6)
            wmn_txt.value = round(wmn.value, 2)
            wmx_txt.value = round(wmx.value, 2)

        def _refresh_replica(*_):
            cond, T = wc.value, int(wT.value)
            opts = _replica_options(cond, T)
            _w_replica.options = opts
            if opts and opts[0][1] is not None:
                _w_replica.value = opts[0][1]

        def _refresh_dropdowns(*_):
            cur_cond = wc.value
            new_cond_opts = _cond_options()
            wc.options = new_cond_opts
            cond_vals = [v for _, v in new_cond_opts]
            wc.value = cur_cond if cur_cond in cond_vals else (new_cond_opts[0][1] if new_cond_opts else None)
            _refresh_T()

        def _refresh_T(*_):
            cond = wc.value
            if cond is None:
                return
            cur_T      = wT.value
            new_T_opts = _T_options(cond)
            wT.options = new_T_opts
            T_vals     = [v for _, v in new_T_opts]
            wT.value   = cur_T if cur_T in T_vals else (new_T_opts[0][1] if new_T_opts else None)
            _load_state_into_widgets()
            _refresh_replica()

        wc.observe(_refresh_T, names="value")
        wT.observe(_load_state_into_widgets, names="value")
        wT.observe(_refresh_replica, names="value")

        def _on_force_replica(_btn):
            cond, T = wc.value, int(wT.value)
            fname = _w_replica.value
            if fname is None:
                return
            data = all_kk_data.get(cond, {}).get(T)
            if not data:
                return
            best_name = data["records"][data["best_idx"]].path.name
            with out2:
                _clear(wait=True)
                if fname == best_name and not OVERRIDES.get(cond, {}).get(T):
                    print("Already auto-selected; no override needed.")
                    return
                if fname == best_name:
                    OVERRIDES.get(cond, {}).pop(T, None)
                    print(f"Override removed for {_short_cond(cond)} T={T}; auto-selection restored.")
                else:
                    OVERRIDES.setdefault(cond, {})[T] = fname
                    print(f"OVERRIDES['{cond}'][{T}] = '{fname}'")
                    print("Re-run Step 3 (export) to apply.")
            _refresh_replica()
        _w_force.on_click(_on_force_replica)

        def _on_crit_toggle(change):
            global KK_USE_W_CRITERIA
            KK_USE_W_CRITERIA = bool(change["new"])
            w_crit.description = "ceramic W-criteria" if KK_USE_W_CRITERIA else "strict kk>=0.97"
            _reclassify_all()
            chips.value = _status_chips_html()
            _refresh_dropdowns()
            with out2:
                _clear(wait=True)
                g, y, r = _count_status()
                lbl = "ceramic electrolyte (W_re>=0.95 AND W_im>=0.93)" if KK_USE_W_CRITERIA else "strict kk>=0.97"
                print(f"Criterion switched to {lbl}")
                print(f"  {g} GREEN | {y} YELLOW | {r} RED")
        w_crit.observe(_on_crit_toggle, names="value")

        def _on_retest(_btn=None):
            global KK_IQR_FENCE, KK_IQR_WINDOW
            KK_IQR_FENCE  = float(wfence.value)
            KK_IQR_WINDOW = int(wwin.value)
            with out2:
                _clear(wait=True)
                _retest_kk(wc.value, int(wT.value),
                           float(wmn.value) if wmn.value > 0 else None,
                           float(wmx.value) if wmx.value > 0 else None,
                           fence=float(wfence.value), window=int(wwin.value))
            _reclassify_all()
            chips.value = _status_chips_html()
            _refresh_dropdowns()
            with out2:
                g, y, r = _count_status()
                if y + r > 0:
                    print(f"  -> {y} YELLOW, {r} RED remaining. "
                          "Re-run Cell A (Step 1) to refresh the classification table.")
                else:
                    print("  -> All spectra GREEN. "
                          "Re-run Cell A (Step 1) to refresh the table, then go to Step 3.")
        wgo.on_click(_on_retest)
        for _w in (wfence, wwin, wmn_txt, wmx_txt):
            _w.observe(lambda ch: _on_retest(), names="value")

        def _on_apply_preset(_btn):
            global KK_C, KK_IQR_FENCE, KK_IQR_WINDOW, KK_F_MIN_HARD, KK_USE_W_CRITERIA
            p = _KK_PRESETS[wpreset.value]
            KK_C              = p["KK_C"]
            KK_IQR_FENCE      = p["KK_IQR_FENCE"]
            KK_IQR_WINDOW     = p["KK_IQR_WINDOW"]
            KK_F_MIN_HARD     = p["KK_F_MIN_HARD"]
            KK_USE_W_CRITERIA = p["KK_USE_W_CRITERIA"]
            w_crit.value = bool(KK_USE_W_CRITERIA)
            wmn.value    = float(KK_F_MIN_HARD) if KK_F_MIN_HARD is not None else 30.0
            wfence.value = float(KK_IQR_FENCE)
            wwin.value   = int(KK_IQR_WINDOW)
            with out2:
                _clear(wait=True)
                print(f"Loaded preset '{wpreset.value}':")
                for k, v in p.items():
                    print(f"  {k} = {v}")
                print("Re-run Step 1 to apply to the full batch.")
        w_apply.on_click(_on_apply_preset)

        _load_state_into_widgets()
        _refresh_replica()
        _display(W.VBox([
            W.HBox([wc, wT, w_crit]),
            W.HBox([wmn, wmn_txt]),
            W.HBox([wmx, wmx_txt]),
            W.HBox([wfence, wwin, wgo]),
            W.HBox([_w_replica, _w_force]),
            W.HBox([wpreset, w_apply]),
            chips, out2,
        ]))
elif not all_kk_data:
    print("[INFO] No KK data; run Step 1 first.")

## Step 3: Selection summary and export

Compact per-condition table of the selected replicas and writes `stage2_kk.xlsx`
(All + Selected sheets) consumed by Stage 3.

When `FOCUS_T` is set, the export **merges** into the existing file; rows for other
temperatures are preserved. `FOCUS_T = None` → full overwrite (safe for first run).

In [ ]:
# Summary Styler table per condition
from pipeline.utils import merge_sheet_by_T, build_metadata_sheet

for condition, cond_data in all_kk_data.items():
    if not cond_data:
        continue
    print(f"\nCondition: {condition}")

    rows = []
    for T_int in sorted(cond_data.keys(), reverse=True):
        data = cond_data[T_int]
        best = data["kk_results"][data["best_idx"]]
        rec  = data["records"][data["best_idx"]]

        if   best["pass_re"] and best["pass_im"]:     kk_lbl = "✓ PASS"
        elif best["pass_re"] and not best["pass_im"]: kk_lbl = "✗ Re only"
        elif best["pass_im"] and not best["pass_re"]: kk_lbl = "✗ Im only"
        else:                                         kk_lbl = "✗ fail"

        rows.append({
            "T [°C]":    T_int,
            "pO₂ [bar]": round(float(rec.pO2_mean or 0), 4),
            "file":      data["selected_file"],
            "kk_score":  round(best["kk_score"], 3),
            "W_re":      round(best["W_re"], 3),
            "W_im":      round(best["W_im"], 3),
            "KK":        kk_lbl,
            "f_min [Hz]": (round(data["f_min_cut"], 1)
                           if data["f_min_cut"] is not None else ","),
            "f_max [Hz]": (round(data["f_max_cut"], 1)
                           if data["f_max_cut"] is not None else ","),
            "★": "★",
        })

    df_disp = pd.DataFrame(rows)

    def _highlight_kk(val):
        return "background-color: #fff3cd" if "✗" in str(val) else ""

    styler = (
        df_disp.style
        .map(_highlight_kk, subset=["KK"])
        .format({"kk_score": "{:.3f}", "W_re": "{:.3f}", "W_im": "{:.3f}"})
        .set_caption(condition.replace("_", " "))
        .set_table_styles([{
            "selector": "caption",
            "props": "font-size: 11px; font-weight: bold; text-align: left;"
        }])
        .hide(axis="index")
    )
    display(styler)

# Build Metadata DataFrame (Lin-KK fixed parameters; applied to all conditions)
df_meta = build_metadata_sheet(
    sample_id  = sample_id,
    stage_name = "stage2_kk",
    params = {
        "KK_C":              KK_C,
        "KK_MU_TARGET":      KK_MU_TARGET,
        "KK_USE_BINARY_M":   KK_USE_BINARY_M,
        "KK_IQR_FENCE":      KK_IQR_FENCE,
        "KK_IQR_WINDOW":     KK_IQR_WINDOW,
        "KK_F_MIN_HARD":     KK_F_MIN_HARD,
        "KK_F_MAX_HARD":     KK_F_MAX_HARD,
        "acceptance_GREEN":  "kk_score >= 0.97 AND frac_cut <= 0.20",
        "acceptance_YELLOW": "kk_score >= 0.90 AND frac_cut <= 0.40",
        "reference":         "Schoenleber et al., Electrochim. Acta 131 (2014); relaXIS manual v1.31",
    },
)

# Export stage2_kk.xlsx (merge-aware when FOCUS_T is set)
print("\nExporting...")
for condition, cond_data in all_kk_data.items():
    if not cond_data:
        print(f"  [{condition}] No data; skipped.")
        continue

    results_dir = sample_dir / "Results" / condition
    results_dir.mkdir(parents=True, exist_ok=True)
    rows_all, rows_sel = [], []

    for T_int, data in sorted(cond_data.items()):
        for i, (rec, res) in enumerate(zip(data["records"], data["kk_results"])):
            f_min, f_max = compute_frequency_cutoffs(res)
            is_sel = i == data["best_idx"]
            row = {
                "condition": condition, "file": rec.path.name,
                "full_path": str(rec.path), "T_nominal": T_int,
                "T_mean":    round(rec.T_mean, 2) if rec.T_mean is not None else None,
                "pO2_mean":  rec.pO2_mean, "replica": rec.replica,
                "kk_score":  round(res["kk_score"], 4),
                "W_re":      round(res["W_re"], 4),
                "W_im":      round(res["W_im"], 4),
                "pass_re":   res["pass_re"],
                "pass_im":   res["pass_im"],
                "mu":        round(res.get("mu", float("nan")), 3),
                "M":         res.get("M"),
                "max_res_re":    round(float(np.abs(res["res_re"]).max()), 4),
                "max_res_im":    round(float(np.abs(res["res_im"]).max()), 4),
                "cutoff_fence":  round(res.get("cutoff_fence", float("nan")), 4),
                "f_min_cut": f_min, "f_max_cut": f_max, "selected": is_sel,
            }
            rows_all.append(row)
            if is_sel:
                rows_sel.append(row)

    xlsx_path = results_dir / "stage2_kk.xlsx"

    df_all = merge_sheet_by_T(xlsx_path, "All",      pd.DataFrame(rows_all), FOCUS_T)
    df_sel = merge_sheet_by_T(xlsx_path, "Selected", pd.DataFrame(rows_sel), FOCUS_T)
    _export_mode = f"merged T={FOCUS_T}°C" if (FOCUS_T is not None and xlsx_path.exists()) else "full overwrite"

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_all.to_excel(writer,  sheet_name="All",      index=False)
        df_sel.to_excel(writer,  sheet_name="Selected", index=False)
        df_meta.to_excel(writer, sheet_name="Metadata", index=False)

    print(f"  [{condition}]  {len(df_sel)} selected → {xlsx_path.relative_to(NOTEBOOK_DIR)}  [{_export_mode}]")

print("\nExport complete.")
print("→ Next: 03_drt_zarc.ipynb")

**Next step:** run [stage3_drt.ipynb](stage3_drt.ipynb)